# Query Transformation and HyDE — narrative notebook

This notebook imports the canonical reference module `query_transformation_hyde.py` (which *owns*
every number) and walks the topic section by section. In a dual encoder a query is *question-shaped*
and a document is *answer-shaped*: the bare query sits **off the document manifold**, so its nearest
documents are mediocre. **HyDE** generates a hypothetical answer document, embeds *that*, and
retrieves real documents near it — landing back inside the manifold.

We reuse the dense-retrieval finance **document manifold** but build our own off-manifold queries.

In [ ]:
import numpy as np
import query_transformation_hyde as qth

P, sector = qth.document_manifold()
g = qth.query_offset_axis(P)
targets = qth.hallucination_targets(P)
print("documents:", P.shape[0], " dim:", P.shape[1], " sectors:", sector.tolist())

## Movement 1 — the query–document gap

A bare query for company $c$ at distribution-shift angle $\theta$ is
$q_c(\theta)=\operatorname{normalize}(\cos\theta\,u_c+\sin\theta\,g)$, tilted off its answer
direction $u_c$ toward the generic "document-ness" axis $g$ (the corpus centroid). As $\theta$ grows
the bare query loses company specificity and its recall@1 collapses — while HyDE, which ignores the
query's position and synthesizes an on-manifold proxy, recovers the answer at every $\theta$.

In [ ]:
theta = np.radians(qth.THETA_OP_DEG)
bare = qth.bare_recall_at_theta(P, g, theta)
hyde = qth.hyde_recall(P, targets, k=qth.K_INF, p=0.0)
print(f"at theta={qth.THETA_OP_DEG:.0f} deg:  bare recall@1 = {bare:.3f}   HyDE recall@1 = {hyde:.3f}")
print("bare recall@1 vs theta:",
      [round(qth.bare_recall_at_theta(P, g, np.radians(d)), 3) for d in qth.SHIFT_GRID_DEG])

**Collapse anchor.** A *perfect* (infinitely concentrated, faithful) single hypothetical *is*
the gold document, so retrieving with it reproduces the gold document's own retrieval exactly.

In [ ]:
for c in (0, 3, 7):
    s = qth.perfect_hypothetical_retrieval(c, P)
    print(f"company {c}: perfect-hypothetical top-1 = {int(np.argmax(s))}  (== gold {c})")

## Movement 2 — HyDE as a Monte-Carlo estimator: bias vs variance

Averaging $k$ hypothetical embeddings, $\hat h_k=\operatorname{normalize}(\tfrac1k\sum_i h_i)$, is a
Monte-Carlo estimate of the generation distribution's center. With a **faithful** generator the
estimate concentrates on the true answer: recall rises and the angular variance
$1-\langle\hat h_k,\mu\rangle$ falls toward the $1/k$ rate. But when the generator **hallucinates**
on a fraction $p$ of queries, $\hat h_k$ is consistent for the *wrong* center, so recall plateaus at a
ceiling near $1-p$ — averaging cannot fix a biased generator.

In [ ]:
print("recall@1 vs k, by hallucination rate p:")
for p in qth.HALLU_GRID:
    print(f"  p={p}: ", [round(qth.hyde_recall(P, targets, k, p=p), 3) for k in qth.K_GRID],
          f"  ceiling ~ {1-p:.2f}")
print("estimator deficit 1-<hhat_k,mu> (faithful):",
      [round(qth.estimator_deficit(0, P, k), 3) for k in qth.K_GRID])

## Movement 3 — HyDE is the neural generalization of pseudo-relevance feedback

Rocchio/RM3 expand a query toward the centroid of pseudo-relevant *retrieved* documents,
$q'=a\,q+b\cdot\text{centroid}$. HyDE lifts the **same** update into embedding space with the centroid
taken over *generated* hypotheticals. On an off-manifold query the generated centroid wins decisively:
real pseudo-relevance feedback retrieves bad documents, so its centroid is polluted and actively hurts.

In [ ]:
theta = np.radians(qth.THETA_OP_DEG)
print("recall@1 vs alpha (interpolation weight):")
print("  HyDE (generated centroid):", [round(v,3) for v in qth.recall_vs_alpha(P, g, 'hyde', theta)])
print("  real PRF (retrieved centroid):", [round(v,3) for v in qth.recall_vs_alpha(P, g, 'prf', theta)])
print("\nthe imported term-space ancestor (RM3/Rocchio recall@4, improve then drift):")
for r in qth.prf_ancestor_curve():
    print(f"  n_fb={r['n_fb']}: RM3={r['rm3']}  Rocchio={r['rocchio']}")

## Verification

The reference module asserts every claim above; running its harness reproduces the headline numbers
and the baked viz constants.

In [ ]:
qth._run_all()
print()
qth.hyde_demo()